In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from gluonts.dataset.util import to_pandas

from tsfm_lens.chronos2.circuitlens import CircuitLensChronos2
from tsfm_lens.dataset import GiftEvalDataset
from tsfm_lens.metrics import compute_metrics
from tsfm_lens.utils import apply_custom_style, load_dyst_samples, set_seed


In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")


In [ ]:
DEFAULT_CONTEXT_LENGTH = 8192

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
STOR_DIR = os.getenv("STOR", "/work")
DATA_DIR = os.path.join(STOR_DIR, "data")
plot_save_dir = os.path.join("../../figs", "ablations_chronos2", "examples")
os.makedirs(plot_save_dir, exist_ok=True)


### Load Pipeline

In [ ]:
model_id = "amazon/chronos-2"  # replace with desired Chronos-2 checkpoint
context_length = DEFAULT_CONTEXT_LENGTH

pipeline = CircuitLensChronos2(
    model_name=model_id,
    device=device,
)


In [ ]:
num_layers = pipeline.num_layers
num_heads = pipeline.num_heads
print(f"num_layers: {num_layers}, num_heads: {num_heads}")


In [ ]:
model_param_memory_mb = sum(p.numel() * p.element_size() for p in pipeline.model.parameters()) / (1024 * 1024)
print(f"Model parameter memory: {model_param_memory_mb:.2f} MB")


### Load Data

In [ ]:
split_name = "gift-eval"
subsample_interval = 1

if split_name == "base40":
    split_dir = os.path.join(DATA_DIR, split_name)
    system_name = "Thomas"
    system_name_pretty = system_name

    context_length_data = 512
    context_start_time = 2000
    context_end_time = context_start_time + context_length_data

    sample_idx = 0
    selected_dim = [0, 1, 2]

    dyst_coords = load_dyst_samples(system_name, split_dir, one_dim_target=False, num_samples=1)
    dyst_coords = torch.tensor(dyst_coords[sample_idx, selected_dim, :])
    dyst_coords = dyst_coords[:, ::subsample_interval]
    context_target = dyst_coords[:, context_start_time:context_end_time]

    prediction_length = 512
    future_vals = dyst_coords[:, context_end_time : context_end_time + prediction_length]

elif split_name == "gift-eval":
    from itertools import islice

    split_dir = os.path.join(DATA_DIR, split_name)
    system_name = "ett1/H"
    to_univariate, term = False, "medium"
    selected_sample_idx, selected_dim = 1, list(range(7))

    dataset = GiftEvalDataset(name=system_name, term=term, to_univariate=to_univariate, data_dir=split_dir)

    system_name_pretty = f"{system_name.replace('/', '_')}_dim{selected_dim}_sample{selected_sample_idx}"

    prediction_length = dataset.test_data.prediction_length

    context_entry = next(islice(dataset.test_data.input, selected_sample_idx, None))
    future_entry = next(islice(dataset.test_data.label, selected_sample_idx, None))
    context_target = context_entry["target"]
    future_target = future_entry["target"]

    if context_target.ndim == 1:
        context_target, future_target = context_target[None, :], future_target[None, :]
    else:
        context_target, future_target = context_target[selected_dim], future_target[selected_dim]

    orig_context_len = context_target.shape[1]
    truncate_offset = max(0, orig_context_len - DEFAULT_CONTEXT_LENGTH)
    context_target = context_target[:, -DEFAULT_CONTEXT_LENGTH:]
    context_start = context_entry["start"] + truncate_offset

    num_variates = context_target.shape[0]
    fig, axes = plt.subplots(num_variates, 1, figsize=(5, 2 * num_variates), squeeze=False)
    for dim, ax in enumerate(axes.flat):
        to_pandas({"target": context_target[dim], "start": context_start}).plot(
            color="black", linewidth=1, ax=ax, label="Context"
        )
        to_pandas({"target": future_target[dim], "start": future_entry["start"]}).plot(
            color="tab:blue", linewidth=1, ax=ax, label="Ground Truth"
        )
        ax.grid(which="both")
        ax.set_title(f"Dim {dim}")
        ax.legend()
    plt.tight_layout()

    context = torch.tensor(context_target)
    future_vals = torch.tensor(future_target)
    num_nans = np.isnan(context_target).sum() + np.isnan(future_target).sum()
    print(f"Context: {context.shape}, Future: {future_vals.shape}, NaNs: {num_nans}")

# reshape to (batch, time, dim) for pipeline
context = context.transpose(0, 1).unsqueeze(0)
future_vals = future_vals.transpose(0, 1).unsqueeze(0)
context = context.to(device)
future_vals = future_vals.to(device)
print(f"Context reshaped for Chronos2: {context.shape}, Future: {future_vals.shape}")


### Ablations

In [ ]:
import json
from itertools import groupby

ASSETS_DIR = os.path.join("../../", "assets")

heads_to_ablate = json.load(
    open(os.path.join(ASSETS_DIR, f"{pipeline.__class__.__name__}_ablate_for_heads_at_1pp.json"))
)

# chosen_layers = list(range(pipeline.num_layers))
chosen_layers = [4, 5, 6, 7]

heads_to_ablate = [config for config in heads_to_ablate if config[0] in chosen_layers]

num_heads_per_layer_to_skip = 3
layers_to_keep_at_heads1pp = [4]
# layers_to_keep_at_heads1pp = []

if num_heads_per_layer_to_skip == 0:
    layers_to_keep_at_heads1pp = chosen_layers
# heads_to_ablate is a list of lists each [layer, head] (should be a tuple, but we can keep it a list for now)
# for each layer, we skip the last head that has that layer as the first element of the tuple
# so we skip the last head for layer 0, the last head for layer 1, etc.

# Group heads by layer and remove the last num_heads_per_layer_to_skip heads from each layer
# except for layers in layers_to_keep_at_heads1pp

heads_to_ablate = [
    config
    for layer, group in groupby(sorted(heads_to_ablate, key=lambda x: x[0]), key=lambda x: x[0])
    for config in (list(group) if layer in layers_to_keep_at_heads1pp else list(group)[:-num_heads_per_layer_to_skip])
]


In [ ]:
chosen_layers_mlp = [4, 5]
print(f"layers chosen for MLP ablation: {chosen_layers_mlp}")

In [ ]:
pipeline.remove_all_hooks()

# print number of heads ablated per layer
for layer in chosen_layers:
    heads_in_layer = [config[1] for config in heads_to_ablate if config[0] == layer]
    print(f"Layer {layer}: {len(heads_in_layer)} heads")

n_heads_ablated = len(heads_to_ablate)
print(f"{n_heads_ablated} heads_to_ablate: {heads_to_ablate}")

ablations_summary_str = f"ablate_{n_heads_ablated}_heads"

print(f"ablations_summary_str: {ablations_summary_str}")

pipeline.add_ablation_hooks_explicit(
    ablations_types=["head", "mlp"],  # type: ignore
    layers_to_ablate_mlp=chosen_layers_mlp,
    heads_to_ablate=heads_to_ablate,
)

layers_without_heads = list(pipeline.head_ablation_handles.keys())
layers_without_mlps = list(pipeline.mlp_ablation_handles.keys())

ablations_summary_str_suffix = ""
if layers_without_heads and layers_without_mlps:
    # if layers_without_heads != layers_without_mlps:
    #     raise NotImplementedError("Zero-ablation of heads and MLPs in different layers is messier, save for later")
    ablations_summary_str_title = (
        f"Zero-Ablation: Heads in Layers {layers_without_heads}, MLPs in Layers {layers_without_mlps}"
    )
    ablations_summary_str_suffix = f"za_heads_layers_{'-'.join(map(str, layers_without_heads))}-mlps_layers_{'-'.join(map(str, layers_without_mlps))}"
elif layers_without_heads:
    ablations_summary_str_title = f"Zero-Ablation: Heads in Layers {layers_without_heads}"
    ablations_summary_str_suffix = f"za_heads_layers_{'-'.join(map(str, layers_without_heads))}"
elif layers_without_mlps:
    ablations_summary_str_title = f"Zero-Ablation: MLPs in Layers {layers_without_mlps}"
    ablations_summary_str_suffix = f"za_mlps_layers_{'-'.join(map(str, layers_without_mlps))}"
else:
    ablations_summary_str_suffix = ablations_summary_str_title = None

ablations_summary_str += "_" + ablations_summary_str_suffix
print(ablations_summary_str)
print(ablations_summary_str_title)


In [ ]:
layers_without_heads = list(pipeline.head_ablation_handles.keys())
layers_without_mlps = list(pipeline.mlp_ablation_handles.keys())

ablations_summary_str_suffix = ""
if layers_without_heads and layers_without_mlps:
    # if layers_without_heads != layers_without_mlps:
    #     raise NotImplementedError("Zero-ablation of heads and MLPs in different layers is messier, save for later")
    ablations_summary_str_title = (
        f"Zero-Ablation: Heads in Layers {layers_without_heads}, MLPs in Layers {layers_without_mlps}"
    )
    ablations_summary_str_suffix = f"za_heads_layers_{'-'.join(map(str, layers_without_heads))}-mlps_layers_{'-'.join(map(str, layers_without_mlps))}"
elif layers_without_heads:
    ablations_summary_str_title = f"Zero-Ablation: Heads in Layers {layers_without_heads}"
    ablations_summary_str_suffix = f"za_heads_layers_{'-'.join(map(str, layers_without_heads))}"
elif layers_without_mlps:
    ablations_summary_str_title = f"Zero-Ablation: MLPs in Layers {layers_without_mlps}"
    ablations_summary_str_suffix = f"za_mlps_layers_{'-'.join(map(str, layers_without_mlps))}"
else:
    ablations_summary_str_suffix = ablations_summary_str_title = None

ablations_summary_str += "_" + ablations_summary_str_suffix
print(ablations_summary_str)
print(ablations_summary_str_title)

In [ ]:
save_fname = (
    f"predictions_{system_name_pretty}_dim{selected_dim}_start{context_start_time}_sub{subsample_interval}"
    if split_name == "base40"
    else f"predictions_{system_name_pretty}"
)

### Predict and Get Outputs

In [ ]:
context.device

In [ ]:
context.shape

In [ ]:
selected_quantiles = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

In [ ]:
rseed = 123
set_seed(rseed)

pred, quantiles = pipeline.predict(context, prediction_length=prediction_length, quantile_levels=selected_quantiles)

In [ ]:
quantiles.shape

In [ ]:
pred.shape

### Plot Predictions with Ablations

In [ ]:
plot_samples = False

In [ ]:
context_np = context.squeeze(0).cpu().numpy().transpose(1, 0)
context_timesteps = np.arange(context_np.shape[-1])
future_vals_np = future_vals.squeeze(0).cpu().numpy().transpose(1, 0)
future_timesteps = np.arange(context_np.shape[-1], context_np.shape[-1] + prediction_length)


In [ ]:
# Chronos2 returns median predictions as (batch, horizon, variates).
median_pred = pred.detach().cpu().numpy()
if median_pred.ndim == 3 and median_pred.shape[0] == 1:
    median_pred = median_pred[0]
if median_pred.ndim != 2:
    raise ValueError(f"Expected median prediction with 2 dims after squeezing batch, got {median_pred.shape}")
if median_pred.shape[0] == len(future_timesteps):
    median_pred = median_pred.T  # (horizon, variates) -> (variates, horizon)

# Chronos2 returns quantile predictions as (batch, quantiles, horizon, variates).
quantile_preds = quantiles.detach().cpu().numpy()
if quantile_preds.ndim == 4 and quantile_preds.shape[0] == 1:
    quantile_preds = quantile_preds[0]
if quantile_preds.ndim != 3:
    raise ValueError(f"Expected quantile predictions with 3 dims after squeezing batch, got {quantile_preds.shape}")
if quantile_preds.shape[1] == len(future_timesteps):
    quantile_preds = quantile_preds.transpose(
        2, 0, 1
    )  # (quantiles, horizon, variates) -> (variates, quantiles, horizon)
elif quantile_preds.shape[2] != len(future_timesteps):
    raise ValueError(f"Could not identify horizon dimension in quantile predictions: {quantile_preds.shape}")

num_variates = median_pred.shape[0]
num_quantiles = quantile_preds.shape[1]

print(f"num_variates: {num_variates}, num_quantiles: {num_quantiles}")

fig, axs = plt.subplots(num_variates, 1, figsize=(8, 2 * num_variates))
axs = [axs] if num_variates == 1 else axs

for dim, ax in enumerate(axs):
    ax.plot(
        context_timesteps[-min(512, len(context_timesteps)) :],
        context_np[dim][-min(512, len(context_timesteps)) :],
        "k-",
        linewidth=1,
        label="Context",
    )
    ax.plot(future_timesteps, future_vals_np[dim], "k--", linewidth=1, label="Ground Truth", zorder=10)
    for q in range(num_quantiles):
        ax.plot(
            future_timesteps,
            quantile_preds[dim, q],
            linewidth=1,
            color=DEFAULT_COLORS[q % len(DEFAULT_COLORS)],
            alpha=0.5,
            label=f"Quantile {q + 1}" if dim == 0 else None,
        )
    ax.plot(
        future_timesteps,
        median_pred[dim],
        color="tab:blue",
        linewidth=2,
        label="Median",
        zorder=11,
    )
    ax.set_xlabel("Timestep", fontweight="bold")
    ax.set_ylabel(f"Dimension {dim}", fontweight="bold")
    # ax.legend(loc="lower left", frameon=True)

plt.suptitle(system_name, fontweight="bold", fontsize=12, y=1.0)
plt.tight_layout()

save_path = os.path.join(plot_save_dir, "zero_ablations", f"{ablations_summary_str}_{save_fname}.pdf")
print(f"Saving to {save_path}")
os.makedirs(os.path.dirname(save_path), exist_ok=True)
fig.tight_layout()
plt.savefig(save_path)


### Plot Original Predictions

In [ ]:
# pipeline.model.chronos_config.quantiles

In [ ]:
pipeline.remove_all_hooks()
set_seed(rseed)

pred_original, quantiles_original = pipeline.predict(
    context, prediction_length=prediction_length, quantile_levels=selected_quantiles
)


In [ ]:
print(f"pred_original shape: {pred_original.shape}")
print(f"quantiles_original shape: {quantiles_original.shape}")

In [ ]:
median_pred_original = pred_original.detach().cpu().numpy()
if median_pred_original.ndim == 3 and median_pred_original.shape[0] == 1:
    median_pred_original = median_pred_original[0]
if median_pred_original.ndim != 2:
    raise ValueError(f"Expected median prediction with 2 dims after squeezing batch, got {median_pred_original.shape}")
if median_pred_original.shape[0] == len(future_timesteps):
    median_pred_original = median_pred_original.T

quantile_preds_original = quantiles_original.detach().cpu().numpy()
if quantile_preds_original.ndim == 4 and quantile_preds_original.shape[0] == 1:
    quantile_preds_original = quantile_preds_original[0]
if quantile_preds_original.ndim != 3:
    raise ValueError(f"Expected quantile predictions with 3 dims after squeezing batch, got {quantile_preds.shape}")
if quantile_preds_original.shape[1] == len(future_timesteps):
    quantile_preds_original = quantile_preds_original.transpose(2, 0, 1)
elif quantile_preds_original.shape[2] != len(future_timesteps):
    raise ValueError(f"Could not identify horizon dimension in quantile predictions: {quantile_preds_original.shape}")

num_variates = median_pred_original.shape[0]
num_quantiles = quantile_preds_original.shape[1]

print(f"num_variates: {num_variates}, num_quantiles: {num_quantiles}")

fig, axs = plt.subplots(num_variates, 1, figsize=(8, 2 * num_variates))
axs = [axs] if num_variates == 1 else axs

for dim, ax in enumerate(axs):
    ax.plot(
        context_timesteps[-min(512, len(context_timesteps)) :],
        context_np[dim][-min(512, len(context_timesteps)) :],
        "k-",
        linewidth=1,
        label="Context",
    )
    ax.plot(future_timesteps, future_vals_np[dim], "k--", label="Ground Truth", linewidth=1, zorder=10)
    for q in range(num_quantiles):
        ax.plot(
            future_timesteps,
            quantile_preds_original[dim, q],
            linewidth=1,
            color=DEFAULT_COLORS[q % len(DEFAULT_COLORS)],
            alpha=0.5,
            label=f"Quantile {q + 1}" if dim == 0 else None,
        )
    ax.plot(
        future_timesteps,
        median_pred_original[dim],
        color="tab:blue",
        linewidth=2,
        label="Median",
        zorder=11,
    )
    ax.set_xlabel("Timestep", fontweight="bold")
    ax.set_ylabel(f"Dimension {dim}", fontweight="bold")
    # ax.legend(loc="lower left", frameon=True)

plt.suptitle(system_name, fontweight="bold", fontsize=12, y=1.0)
plt.tight_layout()

save_path = os.path.join(plot_save_dir, "original", f"{save_fname}.pdf")
print(f"Saving to {save_path}")
os.makedirs(os.path.dirname(save_path), exist_ok=True)
fig.tight_layout()
plt.savefig(save_path)


### Compute Metrics

In [ ]:
pred_horizon_lst = [64]
for pred_horizon in pred_horizon_lst:
    for dim in range(median_pred.shape[0]):
        print(f"Computing metrics for prediction horizon {pred_horizon} and dimension {dim}")
        metrics_combined = compute_metrics(median_pred_original[dim, :pred_horizon], median_pred[dim, :pred_horizon])
        spearman_corr = metrics_combined["spearman"]
        print(f"Spearman correlation: {spearman_corr}")
